PRÁCTICA 2: Aseguramiento de Calidad de Datos en el Sistema Hospital Salud Vital

In [13]:
from dotenv import load_dotenv
import os
import pandas as pd
from sqlalchemy import create_engine

load_dotenv()

#conexion a bd
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

In [14]:
medicos=pd.read_sql("SELECT * FROM medicos", engine)
medicos

,id_medico,nombre_completo,id_especialidad,correo_electronico,nacionalidad
0,1,Sr(a). Alicia Bonilla,1,sr(a)..bonilla@hospitalvital.mx,Estados Unidoss
1,2,Tomás Velázquez Jaimes,5,tomás.jaimes@hospitalvital.mx,Argentina
2,3,Reina Lorena Angulo,3,reina.angulo@hospitalvital.mx,México
3,4,Soledad Trejo,1,soledad.trejo@hospitalvital.mx,España
4,5,Tomás Hermelinda Rosas Guevara,5,tomás.guevara@hospitalvital.mx,
...,...,...,...,...,...
95,96,Ramón Carrasco Bañuelos,3,ramón.bañuelos@hospitalvital.mx,México
96,97,Ing. Eloy Vega,4,ing..vega@hospitalvital.mx,Alemania
97,98,Dr. Daniel Menchaca,3,dr..menchaca@hospitalvital.mx,Argentina
98,99,Lic. Camilo Flórez,2,lic..flórez@hospitalvital.mx,México


LIMPIEZA DE MEDICOS

DETECCION ERRORES

In [15]:
print(medicos.shape)
print(medicos.isnull().sum())
print(medicos.dtypes)

(100, 5)
id_medico             0
nombre_completo       0
id_especialidad       0
correo_electronico    0
nacionalidad          0
dtype: int64
id_medico             int64
nombre_completo         str
id_especialidad       int64
correo_electronico      str
nacionalidad            str
dtype: object


In [16]:
# Nacionalidades unicas — aquí veo los typos
medicos['nacionalidad'].value_counts(dropna=False) #me doy cuenta que en mexico hay México y Mexcio. Colombia y colmbia, null, Estados Unidos, Estados Unidoss

nacionalidad
México             36
Estados Unidos     16
Argentina          14
Alemania           11
España              8
Colombia            4
Colmbia             3
Mexcio              3
Estados Unidoss     2
null                2
                    1
Name: count, dtype: int64

In [17]:
# Valores 'null' como texto y nacionalidades vacias
medicos[medicos['nacionalidad'].isin(['null', '']) | medicos['nacionalidad'].isnull()]

,id_medico,nombre_completo,id_especialidad,correo_electronico,nacionalidad
4,5,Tomás Hermelinda Rosas Guevara,5,tomás.guevara@hospitalvital.mx,
44,45,Renato Carmona Espino,6,renato.espino@hospitalvital.mx,null
94,95,Óliver Nájera,3,óliver.nájera@hospitalvital.mx,null


In [19]:
mask = medicos['correo_electronico'].str.contains(r'\(|\)|mtro\.|ing\.|dr\.|lic\.|sra\.', case=False, na=False)
medicos[mask]

,id_medico,nombre_completo,id_especialidad,correo_electronico,nacionalidad
0,1,Sr(a). Alicia Bonilla,1,sr(a)..bonilla@hospitalvital.mx,Estados Unidoss
14,15,Lic. Felix Crespo,3,lic..crespo@hospitalvital.mx,Colmbia
20,21,Mtro. Amanda Arteaga,1,mtro..arteaga@hospitalvital.mx,México
22,23,Sr(a). Bianca Prado,2,sr(a)..prado@hospitalvital.mx,México
25,26,Dr. Jos Uribe,3,dr..uribe@hospitalvital.mx,España
27,28,Mtro. Sofía Casanova,5,mtro..casanova@hospitalvital.mx,Estados Unidos
30,31,Ing. Karla Sanches,3,ing..sanches@hospitalvital.mx,España
33,34,Ing. Clara Menchaca,2,ing..menchaca@hospitalvital.mx,Alemania
34,35,Mtro. Dalia Orozco,6,mtro..orozco@hospitalvital.mx,Mexcio
35,36,Mtro. Rosalia Miramontes,5,mtro..miramontes@hospitalvital.mx,México


ERRRO 1 CORREGIDO NACIONALIDADES INCOSISTENTES

In [20]:
medicos['nacionalidad'] = medicos['nacionalidad'].replace({
    'Mexcio': 'México',
    'Colmbia': 'Colombia',
    'Estados Unidoss': 'Estados Unidos'
})

# verificar que ya no existan
medicos['nacionalidad'].value_counts(dropna=False)

nacionalidad
México            39
Estados Unidos    18
Argentina         14
Alemania          11
España             8
Colombia           7
null               2
                   1
Name: count, dtype: int64

In [21]:
# Error 2 — 'null' como texto y cadena vacía → mexico
medicos['nacionalidad'] = medicos['nacionalidad'].replace({'null': 'México', '': 'México'})
#misma consulta que antes pero ya no salieron resultados de nacionalidades con null o cadena vacia
medicos[medicos['nacionalidad'].isin(['null', '']) | medicos['nacionalidad'].isnull()]

,id_medico,nombre_completo,id_especialidad,correo_electronico,nacionalidad
